## DYNAP-SE 1 interactive demo

This demo offers two independed experiments:

1. **Spike-frequency adaptaion (SFA)** in a population of 30 neurons on Core 2 of Chip 1 of the board

2. **Soft Winner-take-all network (sWTA)** on Cores 0 and 1 of Chip 1 of the board

To configure the experiments, simply execute the cells of this notebook one by one, following the additional instructions when needed.

All interaction with the chip should be done with the sliders and buttons directly in this notebook, no parameters in the code should require any changes.


### Step 0: External set-up

**1. Connect DYNAP-SE1 locally.**

**2. Connect the oscilloscope probe to Chip 1 Core 2** of the board

In the code, neuron 30 of the C1c2 will be monitored.



#### *Recommended settings for the oscilloscope:*

**- vertical resultion** - 200mV/ 

**- horizontal resolution** - 200ms/

**- channel offset** - 600mV

**- coupling** - DC

**- trigger** - edge mode; at 1V

### Step 1: Import all needed the modules


In [10]:
import sys
sys.path.append('..')


from network_primitives.wta_clustered import WTAGroup, double2pop_code, pop_code2double, check_cams_per_neuron_clustered_wta
from network_primitives.neuron_pool import InteractiveNeuronPool

import matplotlib.pyplot as plt
import numpy as np
from random import sample
from time import sleep
import matplotlib.image as mpimg

from math import pi

import samna
import dynapse1utils as ut
from netgen import NetworkGenerator

from tools.notebook_gui import make_parameter_slider_box

from live_gui import wta_live_plotter
from live_gui.sliderGui import run_threaded_gui

### Step 2: Open the chip and the live spiking Samna GUI

Execute the cell below. This should take around **1min 40sec**.

**Important:** This section should only be executed **once**. Any re-runs usually fail. It would take a kernel restart to re-open the chip and the Samna GUI.

In [3]:
model, _ = ut.open_dynapse1(gui=False, select_device=False)
ut.open_gui(model)

[0]:  Bus 1 Device 6 Dynapse1DevKit serial_number 00000002
Selected device: 00000002
Sender port: tcp://0.0.0.0:42491
Receiver port: tcp://0.0.0.0:59109
Opened device name: Dynapse1DevKit
SamnaNode ID: 1
PythonNode ID: 2
0 Dynapse1Wrapper created! libcaer init...
Clearing chip 0... DONE.
Clearing chip 1... DONE.
Clearing chip 2... DONE.
Clearing chip 3... DONE.
Visualizer start command:  /home/dzenn/anaconda3/envs/dynapse1/bin/python -c "import samna, samnagui; samnagui.runVisualizer(0.75, 0.75,         'tcp://0.0.0.0:59109', 'tcp://0.0.0.0:42491', 3)"


(<Thread(Thread-4, started 140566926980800)>, 'tcp://0.0.0.0:33391')

### Step 3: Spike frequency adaptation experiment set-up

The code below creates a `NetworkGenerator` instance (the object that keeps track of all on-chip connections, it will be shared for both experiments)

In [4]:
net_gen = NetworkGenerator()

In [ ]:
ut.open_gui(model)

Select **Neuron 30 of Core 2** for monitoring (i.e. neuron 542):

In [7]:
api = model.get_dynapse1_api()

chip_id = 1
neuron_id = 512 + 18

api.monitor_neuron(chip_id, neuron_id)

Create a `DynapseTestGroup` of 16 neurons that will receive input from dedicated FPGA spike generators.

In the code, the parameter file `demo_biases1.txt` is loaded that should silence most of the board as well as the population for our experiment. To see any activity in the **Visualizer** as well as on the oscilloscope, one would have to send inputs.

The slider array below this cell exposes interacative slider controls for individual core biases and a few buttons to send input (single 100Hz pulse, continuous 100Hz stimulation, or ten 100 Hz pulses).
Alternatively, the DC input bias can also be used to send constant input.

Finally, the **`Plot pulse response`** button creates a rasterplot recording of the network response to a 100Hz pulse in the **Log tab** of the Jupyter Lab interface

In [14]:
start_neuron=16
exc_size=16
inh_size=0
exc_cluster_size=1

mode='ff'

input_multiplier=5

sfa_chip_id=1
sfa_core_id=2

A = np.zeros((exc_size,exc_size))

A[2,2]=1

TG_SFA = InteractiveNeuronPool(model,sfa_chip_id,sfa_core_id, start_neuron,exc_size, net_gen=net_gen)

#TG_SFA.set_regular_bump_input(0.5, sigma=10, amplitude=50)

ut.set_parameters_in_txt_file(model, 'parameters/demo_biases1.txt')
ut.set_adaptation_enable(model, sfa_chip_id, sfa_core_id, (7,160))

#ut.set_neuron_adaptation_tau(model, sfa_chip_id, sfa_core_id, (4, 100))
#ut.set_neuron_adaptation_weight(model, sfa_chip_id, sfa_core_id, (7, 110))
#ut.set_neuron_adaptation_gain(model, sfa_chip_id, sfa_core_id, (4, 150))



from ipywidgets import interactive_output, Layout, IntSlider, Button, HBox, VBox, Label, FloatSlider
from tools.spikegen_tools import record_population_response

# Gui Styling
style = {'description_width': 'initial'}
style_labels = {'font_weight' : 'bold'}


# WTA_Gui setup
sfa_label1 = Label('SFA demo setup', style=style_labels)

sfa_label2 = Label('Main neuron parameters', style=style_labels)

sfa_label3 = Label('Adaptation parameters', style=style_labels)


# Create button to open the full bias slider window (if needed)
def show_full_sliders_gui(b):
    run_threaded_gui(model)
    
button_show_bias_gui = Button(description='Full biases window', style=style_labels)
button_show_bias_gui.on_click(show_full_sliders_gui)

# Create button to send a regular rate pulse to demonstrate adaptation

def single_pulse(b):
    rates_vector = np.ones(exc_size)*100
    TG_SFA.send_spike_pulse(rates_vector, duration_s=0.5)

button_single_pulse = Button(description='One 100 Hz Pulse (0.5s)', style=style_labels)
button_single_pulse.on_click(single_pulse)

#def regular_input_on(b):
  #  TG_SFA.set_regular_bump_input(0.5, sigma=10, amplitude=100)
    
#button_constant_input = Button(description='100 Hz constant input', style=style_labels)
#button_constant_input.on_click(regular_input_on)

def ten_pulses(b):
    for _ in range(10):
        single_pulse(b)
        sleep(1)
    
button_ten_pulses = Button(description='10 pulses (100Hz, 0.5s)', style=style_labels)
button_ten_pulses.on_click(ten_pulses)

def record_single_response(b):
    #from IPython.display import clear_output
    #clear_output(wait=True)
    #%matplotlib notebook
    record_population_response(TG_SFA.event_sink_node, TG_SFA.fpga_spike_gen, TG_SFA.size)
    plt.show()
    
button_adaptation_plot = Button(description='Plot pulse response', style=style_labels)
button_adaptation_plot.on_click(record_single_response)

# Input to Exc weight (NMDA)
sfa_inp_bias = ut.get_nmda_weight(model, sfa_chip_id, sfa_core_id, 'index') # get the loaded value to the slider
sfa_slider1 = IntSlider(description='Inp->E (NMDA)', style=style, min=0, max=1811, value=sfa_inp_bias)
sfa_inp_wgt_slider = interactive_output(lambda x: ut.set_nmda_weight(model, sfa_chip_id, sfa_core_id, x, 'index'), {'x':sfa_slider1})
# DC input bias
sfa_neuron_dc_bias = ut.get_neuron_dc(model, sfa_chip_id, sfa_core_id, 'index') # get the loaded value to the slider
sfa_slider2 = IntSlider(description='Core DC input', style=style, min=0, max=1811, value=sfa_neuron_dc_bias)
sfa_ee_wgt_slider = interactive_output(lambda x: ut.set_neuron_dc(model, sfa_chip_id, sfa_core_id, x, 'index'), {'x':sfa_slider2})
# Tau 1 bias
sfa_tau1_bias = ut.get_neuron_tau1(model, sfa_chip_id, sfa_core_id, 'index') # get the loaded value to the slider
sfa_slider3 = IntSlider(description='Tau 1', style=style, min=0, max=1811, value=sfa_tau1_bias)
sfa_ie_wgt_slider = interactive_output(lambda x: ut.set_neuron_tau1(model, sfa_chip_id, sfa_core_id, x, 'index'), {'x':sfa_slider3})
# Refractory period bias
sfa_rfr_bias = ut.get_neuron_refractory_period(model, sfa_chip_id, sfa_core_id, 'index') # get the loaded value to the slider
sfa_slider4 = IntSlider(description='Refractory period', style=style, min=0, max=1811, value=sfa_rfr_bias)
sfa_ie_wgt_slider = interactive_output(lambda x: ut.set_neuron_refractory_period(model, sfa_chip_id, sfa_core_id, x, 'index'), {'x':sfa_slider4})
# Neuron gain bias
sfa_gain_bias = ut.get_neuron_gain(model, sfa_chip_id, sfa_core_id, 'index') # get the loaded value to the slider
sfa_slider5 = IntSlider(description='Neuron gain', style=style, min=0, max=1811, value=sfa_gain_bias)
sfa_ie_wgt_slider = interactive_output(lambda x: ut.set_neuron_gain(model, sfa_chip_id, sfa_core_id, x, 'index'), {'x':sfa_slider5})


# Adaptation weight
sfa_ahp_wgt_bias = ut.get_neuron_adaptation_weight(model, sfa_chip_id, sfa_core_id, 'index') # get the loaded value to the slider
sfa_slider6 = IntSlider(description='SFA weight', style=style, min=0, max=1811, value=sfa_ahp_wgt_bias)
sfa_ie_wgt_slider = interactive_output(lambda x: ut.set_neuron_adaptation_weight(model, sfa_chip_id, sfa_core_id, x, 'index'), {'x':sfa_slider6})
# Adaptation tau
sfa_ahp_tau_bias = ut.get_neuron_adaptation_tau(model, sfa_chip_id, sfa_core_id, 'index') # get the loaded value to the slider
sfa_slider7 = IntSlider(description='SFA tau', style=style, min=0, max=1811, value=sfa_ahp_tau_bias)
sfa_ie_tau_slider = interactive_output(lambda x: ut.set_neuron_adaptation_tau(model, sfa_chip_id, sfa_core_id, x, 'index'), {'x':sfa_slider7})
# Adaptation weight
sfa_ahp_gain_bias = ut.get_neuron_adaptation_gain(model, sfa_chip_id, sfa_core_id, 'index') # get the loaded value to the slider
sfa_slider8 = IntSlider(description='SFA gain', style=style, min=0, max=1811, value=sfa_ahp_gain_bias)
sfa_ie_gain_slider = interactive_output(lambda x: ut.set_neuron_adaptation_gain(model, sfa_chip_id, sfa_core_id, x, 'index'), {'x':sfa_slider8})


SFA_label_box = VBox([sfa_label1, button_show_bias_gui, button_single_pulse, button_ten_pulses])
SFA_main_biases_box = VBox([sfa_label2, sfa_slider1, sfa_slider2, sfa_slider3, sfa_slider4, sfa_slider5])
SFA_sfa_biases_box = VBox([sfa_label3, sfa_slider6, sfa_slider7, sfa_slider8, button_adaptation_plot])

HBox([SFA_label_box, SFA_main_biases_box, SFA_sfa_biases_box], layout=Layout(grid_gap='10px 70px'))

New configuration applied to DYNAP-SE1!


In [20]:
make_parameter_slider_box(model, ["C1c2:ampa_w", "C1c2:ampa_tau", "C1c2:ampa_gain", "C1c2:nmda",
                                  "C1c2:nmda_w", "C1c2:nmda_tau", "C1c2:nmda_gain"])

In [19]:
net_gen.network

Post neuron (ChipId,coreId,neuronId): incoming connections [(preNeuron,synapseType), ...]
C1c2n16: [('C0c3s240', 'AMPA'), ('C0c3s240', 'AMPA'), ('C0c3s224', 'GABA_A'), ('C0c3s224', 'GABA_A')]
C1c2n17: [('C0c3s241', 'AMPA'), ('C0c3s241', 'AMPA'), ('C0c3s225', 'GABA_A'), ('C0c3s225', 'GABA_A')]
C1c2n18: [('C0c3s242', 'AMPA'), ('C0c3s242', 'AMPA'), ('C0c3s226', 'GABA_A'), ('C0c3s226', 'GABA_A'), ('C1c2n18', 'NMDA')]
C1c2n19: [('C0c3s243', 'AMPA'), ('C0c3s243', 'AMPA'), ('C0c3s227', 'GABA_A'), ('C0c3s227', 'GABA_A')]
C1c2n20: [('C0c3s244', 'AMPA'), ('C0c3s244', 'AMPA'), ('C0c3s228', 'GABA_A'), ('C0c3s228', 'GABA_A')]
C1c2n21: [('C0c3s245', 'AMPA'), ('C0c3s245', 'AMPA'), ('C0c3s229', 'GABA_A'), ('C0c3s229', 'GABA_A')]
C1c2n22: [('C0c3s246', 'AMPA'), ('C0c3s246', 'AMPA'), ('C0c3s230', 'GABA_A'), ('C0c3s230', 'GABA_A')]
C1c2n23: [('C0c3s247', 'AMPA'), ('C0c3s247', 'AMPA'), ('C0c3s231', 'GABA_A'), ('C0c3s231', 'GABA_A')]
C1c2n24: [('C0c3s248', 'AMPA'), ('C0c3s248', 'AMPA'), ('C0c3s232', 'GABA_

In [18]:
TG_SFA.set_matrix(A)

Netgen updated. Connections added: 1, connections removed: 0
Matrix update time:  0.01821446418762207
New configuration applied to DYNAP-SE1!


In [2]:
from tools.spikegen_tools import record_population_response

### Step 4: Soft Winner-take-all network experiment

The cell below configures the sWTA network (**152 excitatory** neurons in **clusters of 8**; **20 inhibitory** neurons; with **80% of EE connectivity density** within the clusters, **50% excitatory density with the nearest neighbouring clusters**).

**Network parameters** are **adjustable** - re-run this cell with a different network size and the cell below - and controls and the visualization should dynamically reconfigure for the new parameters.

In [21]:
start_neuron=16
exc_size=152

chip_id=1
core_id=0
mode='clustered_WTA'
# mode='ff'
exc_cluster_size=8
inh_cluster_size=4

inh_size=20
connectivity={'wta_structure': 'global_inh',
                'ee_recurrent': 0.8,
                'ee_lateral':0.5,
                'ee_lateral_2nd':0.0,
                'ee_lateral_3rd':0.0,
                'ei': 0.2,
                'ie': 0.2,
                'ii': 0.0,
                'ee_global':0.0,
                'ei_global':0.0,
                'exc_global_size': 0,
                'start_neuron_inh' : 16,
                'edge_wrap_around' : True}


input_multiplier=5
clustered_input = True
allow_self_exc = False
debug = False

# check_cams_per_neuron_clustered_wta(exc_cluster_size, inh_size, exc_size/exc_cluster_size, connectivity, input_multiplier, print_detailed=True)

ut.set_parameters_in_txt_file(model, 'parameters/demo_biases1.txt')

TG = WTAGroup(model, start_neuron,exc_size,inh_size,chip_id,core_id,mode,core_id+1, net_gen, exc_cluster_size,inh_cluster_size,connectivity,input_multiplier,clustered_input,allow_self_exc,debug=debug)
TG.set_bump_input(0.4, sigma=0.15)

Exc cluster neuron CAMs:
-> Input:  5
-> EE recurrent:  6
-> EE lateral:  8
-> IE:  4
Total:  23

Inh cluster neuron CAMs:
-> EI:  30
-> II recurrent:  0
Total:  30
model ok
netgen ok
SynTypes ok
[ 16  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33
  34  35  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51
  52  53  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69
  70  71  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87
  88  89  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105
 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123
 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141
 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159
 160 161 162 163 164 165 166 167  16  17  18  19  20  21  22  23  24  25
  26  27  28  29  30  31  32  33  34  35]
New configuration applied to DYNAP-SE1!


### Experiment interface:

The sliders block below is organized in three columns.

**1.** **Interactive buttons**

- **`Live WTA plot`** button opens an additional window with live raster and rate plots for the sWTA. **Note: it may take a few seconds to open. If it doesn't - check if another window is aleady opened, that can create glitches**
- **`Full biases window`** button opens an additional TKinter window with the full list of biases (for expert tuning). **Make sure to use Chip 1 subpage (Chip 0 is the default view)**

**2.** **Input controls** - this section configures steady Poisson spiking inputs. One or two bump-shaped inputs are possible, with adjustable widths. Touch the respective sliders to activate the respective input mode.

**3.** **Weight biases** - the sliders for the four weight groups are provided: Inp->E, E->E, E->I and I->E. Adjust the time constants of synaptic gains use the **`Full biases window`**. In that window, the button **Save biases** can be used to save the most recent configuraion.


### What to do in the experiment:

With the default parameters, the activity bump in the network should already form.

**0. Open the GUI** by pressing the **`Live WTA plot`** buttons.

**1. Move the input bump**, explore how the activity follows it using the **Bump Position** slider.

**2. Reduce the Inp->E wegights** to detach the bump activity from the input. The activity bump should start drifting. A slight increase in the **E->E weight** might be needed to maintain the mean firing rate at 40-50Hz.

**3. Try the double input**, explore how the network activity selectively amplifies one of the inputs (or does not, depending on the excitatory-inhibitory balance defined by the weights). Adjust the input bump widths for clarity if needed using the **Sigma** slider. Adjust the input bump amplitudes using the **Amp1** and **Amp2** sliders.


In [22]:
from ipywidgets import interactive_output, Layout, IntSlider, Button, HBox, VBox, Label, FloatSlider

# Gui Styling
style = {'description_width': 'initial'}
style_labels = {'font_weight' : 'bold'}

#AHP_box = HBox([input_buttons_box, inputs_sliders_box, bias_sliders_box], layout=Layout(grid_gap='10px 70px'))

# WTA_Gui setup
label1 = Label('WTA setup', style=style_labels)
label1_1 = Label('(might take a few seconds)')

# Double bump controls section setup - goes first so that the single bump section is triggered last
label3 = Label('Double bump input', style=style_labels)
slider3 = FloatSlider(description='Bump 1 Position', style=style, min=0, max=1, step=0.01, value=0.2)
slider4 = FloatSlider(description='Bump 2 Position', style=style, min=0, max=1, step=0.01, value=0.8)
slider5 = FloatSlider(description='Sigma', style=style, min=0, max=1, step=0.01, value = 0.15)
slider6 = IntSlider(description='Amp1 (Hz)', style=style, min=0, max=150, value=100, orientation='vertical')
slider7 = IntSlider(description='Amp2 (Hz)', style=style, min=0, max=150, value=70, orientation='vertical')
amp_sliders_box = HBox([slider6, slider7])

double_input_sliders = interactive_output(lambda x,y, sigma, a1, a2: TG.set_double_bump_input(x, y,
                                                                                       amplitude1=a1,
                                                                                       amplitude2=a2,
                                                                                       sigma=sigma),
                                   {'x':slider3, 'y':slider4, 'sigma':slider5, 'a1':slider6, 'a2':slider7})

# Single bump section setup - goes last so that it is triggered last
label2 = Label('Single bump input', style=style_labels)
slider1 = FloatSlider(description='Bump Position', style=style, min=0, max=1, step=0.01, value=0.4)
slider2 = FloatSlider(description='Sigma', style=style, min=0, max=1, step=0.01, value = 0.15)

input_slider = interactive_output(lambda x, sigma: TG.set_bump_input(x, sigma=sigma), {'x':slider1, 'sigma':slider2})

# Allow opening live plotting using the button
def show_wta_gui(b):
    plt.ioff() #test this
    wta_live_plotter.run_plotting_thread(TG.running_raster_sink_node, 20, [TG.exc_population_ids[:,2], TG.inh_population_ids[:,2]], TG=TG)

button_show_wta_gui = Button(description='Live WTA plot', style=style_labels)
button_show_wta_gui.on_click(show_wta_gui)

def show_full_sliders_gui(b):
    run_threaded_gui(model)
    
button_show_bias_gui = Button(description='Full biases window', style=style_labels)
button_show_bias_gui.on_click(show_full_sliders_gui)

# Biases section

ut.set_parameters_in_txt_file(model, 'parameters/demo_biases1.txt') # reload default biases

label4 = Label('Weight biases', style=style_labels)

# Input to Exc clusters weight (NMDA)
inp_bias = ut.get_nmda_weight(model, chip_id, core_id, 'index') # get the loaded value to the slider
slider8 = IntSlider(description='Inp->E (NMDA)', style=style, min=0, max=1811, value=inp_bias)
inp_wgt_slider = interactive_output(lambda x: ut.set_nmda_weight(model, chip_id, core_id, x, 'index'), {'x':slider8})
# Exc to Exc clusters weight (AMPA)
ee_bias = ut.get_ampa_weight(model, chip_id, core_id, 'index') # get the loaded value to the slider
slider9 = IntSlider(description='E->E (AMPA)', style=style, min=0, max=1811, value=ee_bias)
ee_wgt_slider = interactive_output(lambda x: ut.set_ampa_weight(model, chip_id, core_id, x, 'index'), {'x':slider9})
# Exc to Inh clusters weight (AMPA)
ei_bias = ut.get_ampa_weight(model, chip_id, core_id+1, 'index') # get the loaded value to the slider
slider10 = IntSlider(description='E->I (AMPA)', style=style, min=0, max=1811, value=ei_bias)
ei_wgt_slider = interactive_output(lambda x: ut.set_ampa_weight(model, chip_id, core_id+1, x, 'index'), {'x':slider10})
# Inh to Exc clusters weight (GABA B)
ie_bias = ut.get_gaba_b_weight(model, chip_id, core_id, 'index') # get the loaded value to the slider
slider11 = IntSlider(description='I->E (GABA B)', style=style, min=0, max=1811, value=ie_bias)
ie_wgt_slider = interactive_output(lambda x: ut.set_gaba_b_weight(model, chip_id, core_id, x, 'index'), {'x':slider11})

bias_sliders = VBox([slider8, slider9, slider10, slider11])


input_buttons_box = VBox([label1, button_show_wta_gui, label1_1, button_show_bias_gui])
inputs_sliders_box = VBox([label2, slider1, slider2, double_input_sliders, label3, slider3,
                           slider4, slider5, amp_sliders_box])
bias_sliders_box = VBox([label4, bias_sliders])
#WTA_box = 
HBox([input_buttons_box, inputs_sliders_box, bias_sliders_box], layout=Layout(grid_gap='10px 70px'))
#VBox([AHP_box, WTA_box])

/home/dzenn/Documents/dynap-se1_latest/dynap-se1/live_gui/wta_live_plotter.py:152: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  self.activity_ax.set_xlim((self.activity_x.min(), self.activity_x.max()))
